In [ ]:
import pandas as pd

# Load the Excel file into a dataframe
raw_df = pd.read_excel('lending_clubFull_Data_Set.xlsx')

## Explore the Data

First, let's examine the structure and available columns in the dataset.

In [ ]:
# Examine the shape and columns
print(f"Dataset shape: {raw_df.shape}")
print(f"\nColumns available: {raw_df.columns.tolist()}")
print(f"\nTarget variable (loan_status) values:")
print(raw_df['loan_status'].value_counts())

## Feature Selection and Preparation

We'll select features that are good predictors of loan default. These include employment information, credit history, and debt metrics.

In [ ]:
# Select key features for prediction
selected_features = [
    'loan_status',
    'emp_length',
    'home_ownership',
    'dti',
    'delinq_2yrs',
    'inq_last_6mths',
    'revol_util',
    'acc_now_delinq'
]

# Create a working dataframe with selected features
df = raw_df[selected_features].copy()

# Remove loans that don't meet credit policy
df = df[~df['loan_status'].str.contains('Does not meet the credit policy', na=False)]

# Create binary target: 1 if Default, Charged Off, or Late (31-120 days), else 0
df['loan_status'] = df['loan_status'].apply(
    lambda x: 1 if x in ['Default', 'Charged Off', 'Late (31-120 days)'] else 0
)

print(f"Target variable distribution:")
print(df['loan_status'].value_counts())
print(f"Default rate: {df['loan_status'].mean():.2%}")

## Data Cleaning

Clean and convert features to numeric values.

In [ ]:
# Clean home_ownership: keep only OWN, MORTGAGE, RENT
for keyword in ['ANY', 'OTHER', 'NONE']:
    df = df[~df['home_ownership'].str.contains(keyword, na=False)]

df['home_ownership'] = df['home_ownership'].replace({'OWN': 1, 'MORTGAGE': 0, 'RENT': -1})

# Convert emp_length to numeric
df['emp_length'] = df['emp_length'].fillna(0)
replacement_map = {
    '< 1 year': 0.5,
    '1 year': 1,
    '10+ years': 10
}
for i in range(2, 10):
    replacement_map[f'{i} years'] = i

df['emp_length'] = df['emp_length'].replace(replacement_map)
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')

# Remove rows with missing values in critical columns
for col in ['home_ownership', 'dti', 'revol_util']:
    df = df.dropna(subset=[col])

# Fill remaining columns with 0
for col in ['delinq_2yrs', 'inq_last_6mths', 'acc_now_delinq']:
    df[col] = df[col].fillna(0)

print(f"Dataset shape after cleaning: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

## Train/Test Split and Preprocessing

Split data into training and test sets, and standardize the features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Prepare features and target
X = df.drop('loan_status', axis=1)
y = df['loan_status']

# Split into training (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Features: {X_train.columns.tolist()}")

## Train Predictive Model

Train a Logistic Regression model to predict loan default status.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions on test set
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Performance Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f} (of predicted defaults, how many are actual defaults)")
print(f"  Recall:    {recall:.4f} (of actual defaults, how many we catch)")
print(f"  F1-Score:  {f1:.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"  True Negatives:  {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives:  {cm[1, 1]}")

# Show feature importance (coefficients)
print("\nFeature Importance (Logistic Regression Coefficients):")
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(feature_importance)

## Predictions on New Data

Use the trained model to predict loan default status probability.

In [ ]:
# Example: Get default probability for a few test samples
print("Sample Predictions (Default Probability):")
for i in range(min(5, len(X_test))):
    prob = y_pred_proba[i]
    actual = y_test.iloc[i]
    predicted = y_pred[i]
    print(f"Sample {i+1}: Prob(Default)={prob:.3f}, Predicted={predicted}, Actual={actual}")